In [1]:
!pip install datasets -q

In [10]:
import random
import re
from collections import defaultdict, Counter

import matplotlib.pyplot as plt
from datasets import load_dataset

In [3]:
print("Loading TinyStories dataset...")

dataset = load_dataset("roneneldan/TinyStories")

print(dataset)

stories = dataset["train"]["text"][:5000]

print(f"\nLoaded {len(stories)} stories.")

Loading TinyStories dataset...


README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…): reconstructing file:   0%|          |  0.00B /  249MB            

data/train-00000-of-00004-2d5a1467fff108(…): downloading bytes:           |  0.00B            

data/train-00001-of-00004-5852b56a2bd28f(…): reconstructing file:   0%|          |  0.00B /  248MB            

...

data/validation-00000-of-00001-869c898b5(…): reconstructing file:   0%|          |  0.00B / 9.99MB            

data/validation-00000-of-00001-869c898b5(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})

Loaded 5000 stories.


In [4]:
#combining stories
text = " ".join(stories)

print("Total characters:", len(text))
print(text[:500])

Total characters: 4181739
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them b


In [5]:
text = text.lower()

text = re.sub(r"[^a-zA-Z0-9\s']", " ", text)

text = re.sub(r"\s+", " ", text)

words = text.split()

print("Total Words:", len(words))
print(words[:30])

Total Words: 811603
['one', 'day', 'a', 'little', 'girl', 'named', 'lily', 'found', 'a', 'needle', 'in', 'her', 'room', 'she', 'knew', 'it', 'was', 'difficult', 'to', 'play', 'with', 'it', 'because', 'it', 'was', 'sharp', 'lily', 'wanted', 'to', 'share']


In [11]:
# Build Bigram Markov Chain

transition_counts = defaultdict(Counter)

for i in range(len(words) - 2):

    state = (words[i], words[i + 1])

    next_word = words[i + 2]

    transition_counts[state][next_word] += 1

print("Unique States:", len(transition_counts))

Unique States: 147205


In [12]:
# Convert Counts into Probabilities

transition_probabilities = {}

for state, counter in transition_counts.items():

    total = sum(counter.values())

    transition_probabilities[state] = {

        word: count / total

        for word, count in counter.items()

    }

print("Transition probability model created successfully.")

Transition probability model created successfully.


In [13]:
# Display Transition Probabilities

sample_state = ("there", "was")

if sample_state in transition_probabilities:

    print(f"Transition probabilities for {sample_state}\n")

    for word, prob in sorted(

        transition_probabilities[sample_state].items(),

        key=lambda x: x[1],

        reverse=True

    )[:10]:

        print(f"{word:<15} {prob:.4f}")

else:

    print("State not found.")

Transition probabilities for ('there', 'was')

a               0.9359
an              0.0342
no              0.0068
nothing         0.0054
something       0.0034
only            0.0017
one             0.0014
nobody          0.0009
to              0.0009
the             0.0009


In [14]:
def generate_text(start_words, length=60):

    state = tuple(start_words.lower().split())

    if len(state) != 2:

        raise ValueError("Please enter exactly TWO starting words.")

    if state not in transition_probabilities:

        return "Starting state not found."

    generated = list(state)

    for _ in range(length):

        transitions = transition_probabilities.get(state)

        if not transitions:

            break

        next_words = list(transitions.keys())

        probabilities = list(transitions.values())

        next_word = random.choices(

            next_words,

            weights=probabilities,

            k=1

        )[0]

        generated.append(next_word)

        state = (state[1], next_word)

    return " ".join(generated)

In [15]:
# Predict Next Words

def predict_next_words(start_words, top_n=10):

    state = tuple(start_words.lower().split())

    if state not in transition_probabilities:

        print("State not found.")

        return

    predictions = sorted(

        transition_probabilities[state].items(),

        key=lambda x: x[1],

        reverse=True

    )[:top_n]

    print(f"\nTop {top_n} Predictions\n")

    for word, prob in predictions:

        print(f"{word:<15} {prob:.4f}")

In [16]:
predict_next_words("once there")


Top 10 Predictions

was             0.9613
were            0.0305
lived           0.0081


In [18]:
print(generate_text("the king", 80))

the king saw that bunny won't steal anything else there was a tall tree the little bird knew it paying off to sleep now the bird felt tired and wet give it to pretend to drive her mom gave her a kiss she said lily took a sip and it broke into a puddle let's go to the table and some dots she was playing he met new friends one day lily went inside to tell ben said we just want to


In [19]:
print(generate_text("once there", 60))

once there was a bad noise coming from the people built the coolest tower ever lily exclaimed yes lily we can escape through this country max barked and licked the bone it was all dry and comfortable her mother laughed and laughed then she noticed it was so happy that it was a father and he said let s eat it but


In [20]:
print(generate_text("there was", 100))

there was a grumpy dog his name was john and his mom went to visit her favorite was anna who had spoken it was part of the pinch the animals wanted to play with the big kids helped him and went to her bed it was time to go so off they went to a place where people went to the other was big and ready to go home suddenly he heard a little girl saw a bush it was the best cake they also made a funny film about a new dress there were two green spiders crawling on the other
